[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-02.ipynb)

# 2주차 실습: 학습 루프와 텐서 연산

**목표.** 손실을 계산하고 역전파로 가중치를 갱신하는 **학습 루프 한 바퀴**를 직접 돌려 손실이 내려가는 것을 확인한다. 표기(방언 vs 표준어)가 토크나이저에서 다른 비용으로 이어지는 것도 실측한다.

이 노트북은 이후 모든 주차 실습이 재사용하는 **기본 학습 루프**다. 처음부터 끝까지 한 번 돌려서 감을 잡는 것이 이번 주 목표다.


## 0. 준비

아래 셀을 실행해 필요한 라이브러리를 설치한다. Colab 무료 런타임이면 이미 설치돼 있을 수 있지만, 안전하게 한 번 실행한다.


In [ ]:
# 필요한 것 설치 (Colab에서 한 번만)
!pip -q install transformers datasets


## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.


### 1-1. 텐서와 자동미분

2주차 온라인에서 본 **자동미분**의 동작을 숫자로 확인한다. `requires_grad=True` 텐서에 연산을 걸면 `backward()` 호출 시 그래디언트가 채워진다.


In [ ]:
import torch

# x 에 대해 loss 가 미분 가능한지 확인한다
x = torch.tensor(3.0, requires_grad=True)
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

loss = (w * x + b - 10) ** 2   # w*x + b 가 10 에 가까워지도록 하는 손실
loss.backward()

print(f"loss = {loss.item():.4f}")
print(f"d(loss)/dw = {w.grad.item():.4f}")
print(f"d(loss)/dx = {x.grad.item():.4f}")
print(f"d(loss)/db = {b.grad.item():.4f}")


### 1-2. 학습 루프 한 바퀴

이제 **순전파 - 손실 - 역전파 - 갱신** 순서를 한 바퀴 돌린다. 가중치가 한 번 갱신된 뒤 손실이 어떻게 변하는지 본다.


In [ ]:
import torch

# 데이터: x 가 주어졌을 때 3*x + 1 에 가까운 값을 내는 선형 모델을 맞춘다
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[4.0], [7.0], [10.0], [13.0]])  # 정답: 3*x + 1

w = torch.tensor([[0.5]], requires_grad=True)
b = torch.tensor([[0.0]], requires_grad=True)

def predict(x):
    return x @ w + b

def mean_squared_error(pred, target):
    return ((pred - target) ** 2).mean()

lr = 0.01

for step in range(200):
    pred = predict(x)
    loss = mean_squared_error(pred, y)
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    if step % 40 == 0:
        print(f"step {step:3d}  loss = {loss.item():.6f}")

print(f"학습 후 w = {w.item():.3f}, b = {b.item():.3f}  (정답: w=3, b=1)")


### 1-3. 한국어 텍스트 분류 모델 로딩

2주차 온라인에서 본 **사전학습 모델 로딩**이다. 작은 한국어 분류 모델을 불러온다. 3주차에서 이 모델에 어댑터(LoRA)를 붙일 때 같은 코드를 쓴다.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "monologg/koelectra-base-v3-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print(f"모델 전체 파라미터 수: {model.num_parameters():,}")


## 2. 한 지점만 바꿔 보기

아래 셀의 `# TODO` 로 표시된 **한 곳만** 바꾸고 다시 실행하세요.

> 바꾸기 전 결과를 먼저 확인해 두면 무엇이 달라졌는지 비교할 수 있습니다.


In [ ]:
import torch

# x 가 주어졌을 때 3*x + 1 에 가까운 값을 내는 모델 (1-2 와 동일)
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[4.0], [7.0], [10.0], [13.0]])

w = torch.tensor([[0.5]], requires_grad=True)
b = torch.tensor([[0.0]], requires_grad=True)

def predict(x):
    return x @ w + b

def mean_squared_error(pred, target):
    return ((pred - target) ** 2).mean()

def mean_absolute_error(pred, target):
    return (pred - target).abs().mean()

# TODO: 이 한 줄의 mean_squared_error를 제공된 mean_absolute_error로 교체
loss_fn = mean_squared_error

lr = 0.01
for step in range(200):
    pred = predict(x)
    loss = loss_fn(pred, y)
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    if step % 40 == 0:
        print(f"step {step:3d}  loss = {loss.item():.6f}")
print(f"학습 후 w = {w.item():.3f}, b = {b.item():.3f}  (정답: w=3, b=1)")


## 3. 실무: 방언·표준어 표기와 토크나이저 비용

[실무] 같은 뜻을 담는 데 드는 토큰 비용이 **표기에 따라** 달라지는지 실측한다. 사전학습 토크나이저는 표준어 말뭉치로 학습됐으므로, 방언 표기(제주어)는 같은 뜻의 표준어보다 **토큰 수가 더 많아질** 가능성이 크다.


### 3-1. 먼저 교수 작성 예시로 감 잡기

- 아래 세 대응 문장은 교수 작성 예시, AI Hub에서 추출한 실제 샘플 아님
- 토크나이저 출력은 직접 실행해 관찰하되 실제 제주 말뭉치 전체의 경향으로 일반화하지 않기
- 실제 데이터 파일이 없어도 이 예시는 수정 없이 실행


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("monologg/koelectra-base-v3-discriminator")

# 교수 작성 비교 예시. AI Hub 샘플 추출 결과가 아님
pairs = [
    ("혼저 옵서예.", "어서 오세요."),        # 방언 / 표준어
    ("고맙수다", "고맙습니다"),
    ("하르방", "할아버지"),
]

for dialect, standard in pairs:
    d_tok = tokenizer.encode(dialect, add_special_tokens=False)
    s_tok = tokenizer.encode(standard, add_special_tokens=False)
    print(f"방언  [{dialect}] -> {len(d_tok)} 토큰 : {tokenizer.convert_ids_to_tokens(d_tok)}")
    print(f"표준  [{standard}] -> {len(s_tok)} 토큰 : {tokenizer.convert_ids_to_tokens(s_tok)}")
    print()


### 3-2. AI Hub 제주 JSON의 준비된 자동 로더

- [공식 데이터 설명](https://www.aihub.or.kr/aihubdata/data/view.do?dataSetSn=121)에서 `utterance` 배열의 `dialect_form`, `standard_form` 필드 확인
- 이 저장소에는 실제 AI Hub 파일 미포함. 인증된 다운로드 접근과 실제 파일에서의 추출은 아직 검증하지 않음
- 이용 가능한 실제 JSON이 있으면 작업 폴더의 `aihub-jeju-json/`에 파일로 배치. 다운로드·업로드는 자료 준비이며 코드·리스트 작성 요구 없음
- 아래 완성된 로더는 해당 폴더만 탐색하여 방언·표준어 문자열 쌍을 자동 추출. 학생이 `data`를 채우거나 로더를 구현하지 않음
- 파일 미확보 또는 유효한 쌍이 없으면 실제 자료 측정 미실행을 표시하고 끝까지 실행. 교수 작성 예시를 AI Hub 실측으로 대체하지 않음
- 화자 메타데이터·문장 원문 출력 없음. 원본 자료 제출·재배포는 이용조건 별도 확인


In [ ]:
# 공식 공개 schema에 맞춘 교수 제공 로더. 코드 수정 불필요.
import json
from pathlib import Path

source_dir = Path("aihub-jeju-json")
json_files = sorted(source_dir.rglob("*.json"))[:20] if source_dir.is_dir() else []
data = []
skipped_files = 0
for file in json_files:
    try:
        payload = json.loads(file.read_text(encoding="utf-8-sig"))
    except (OSError, UnicodeError, json.JSONDecodeError):
        skipped_files += 1
        continue
    pending = [payload]
    while pending and len(data) < 100:
        item = pending.pop()
        if isinstance(item, dict):
            utterances = item.get("utterance", [])
            if isinstance(utterances, list):
                for utterance in utterances:
                    if not isinstance(utterance, dict):
                        continue
                    dialect = utterance.get("dialect_form")
                    standard = utterance.get("standard_form")
                    if isinstance(dialect, str) and isinstance(standard, str) and dialect.strip() and standard.strip():
                        data.append((dialect.strip(), standard.strip()))
                        if len(data) >= 100:
                            break
            pending.extend(value for key, value in item.items() if key != "utterance" and isinstance(value, (dict, list)))
        elif isinstance(item, list):
            pending.extend(reversed(item))
    if len(data) >= 100:
        break
print("자료 종류: 실제 파일이 있을 때만 AI Hub JSON 측정")
print("탐색 파일 수:", len(json_files), "읽기 실패 파일 수:", skipped_files, "추출 대응쌍 수:", len(data))
if not data:
    print("실제 AI Hub 자료 미확보 또는 유효한 대응쌍 없음. 실제 말뭉치 측정 미실행.")
    print("3-1 교수 작성 예시의 출력과 실제 말뭉치 측정을 구분하여 기록하세요.")
else:
    print("앞의 최대20쌍 토큰 수 (최대20파일·100쌍 한정, 전체 말뭉치 대표성 보장 없음):")
    for dialect, standard in data[:20]:
        d_n = len(tokenizer.encode(dialect, add_special_tokens=False))
        s_n = len(tokenizer.encode(standard, add_special_tokens=False))
        print(f"  방언 {d_n:2d} 토큰 | 표준 {s_n:2d} 토큰 | 차이 {d_n - s_n:+d}")


## 4. 확인 질문

1. 2번에서 손실 함수를 바꿨을 때 학습 곡선(손실 감소 속도·마지막 값)은 어떻게 달라졌나요? 왜 그렇게 됐다고 생각하나요?
2. 학습 루프에서 **역전파(`loss.backward()`)** 와 **갱신(`w -= lr*w.grad`)** 이 각각 무엇을 하는지 한 문장씩 설명하세요.
3. 교수 작성 예시에서 방언 표기가 표준어보다 토큰 수가 늘어난 예가 있었나요? 실제 AI Hub 자료는 측정했는지, 미확보였는지 구분해 적으세요. 같은 뜻을 담는 데 왜 토큰 비용이 달라지는지 토크나이저의 학습 방식과 연결지어 설명하세요.

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.


*(여기에 답을 적으세요)*


## 5. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/deepnlp-2026)의 `assignments/week-02/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.
